In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import numpy as np
import pandas as pd

In [ ]:
# 6.3 기본 Skip-gram 클래스
# 단순히 입력 단어와 주변 단어를 룩업 테이블에서 가져와서 내적을 계산한 다음, 손실 함수를 통해 예측 오차를 최소화하는 방식

class VanillaSkipgram(nn.Module) :
  def __init__(self, vocab_size, embedding_dim) :
    super().__init__()
    self.embedding_dim = nn.Embedding(
        num_embeddings = vocab_size, # 단어 사전의 크기(이산 변수의 개수)
        embedding_dim = embedding_dim, # 임베딩 벡터의 크기(임베딩 벡터의 차원 수)
        padding_idx=None, # 입력 문장을 일정한 길이로 맞춤
        max_norm=None, # 임베딩 벡터의 최대 크기(이상이면 잘라냄)
        norm_type=2.0 # 임베딩 벡터의 크기 제한
        # 노름 타입이 1면 1.1정규화, 2이면 1.2 정규화 방식
    )
    self.linear = nn.Linear(
      in_features = embedding_dim,
      out_features=vocab_size
    )

def foward(self, input_ids) :
  embeddings = self.embedding(input_ids)
  output = self.linear(embeddings)
  return output

In [ ]:
pip install Korpora

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [Korpora]


In [ ]:
pip install konlpy

In [ ]:
from Korpora import Korpora
from konlpy.tag import Okt

In [ ]:
# 6.4 영화 리뷰 데이터세트 전처리

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

In [ ]:
# 6.5 단어 사전 구축
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=["<unk>"])
# <unk>는 OOV에 대응하기 위한 토큰으로 단어 사전에 없는 모든 단어 대응
# 단어 사전의 최대 길이를 5000으로 설정 + speical token 1개로 총 단어 사전 5,001개
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [ ]:
# 6.6 Skip-gram의 단어 쌍 추출
def get_words_pairs(tokens, window_size) : # 토큰을 입력받아 skip-gram 모델의 입력 데이터로 사용할 수 있게 전처리
  pairs = [  ]
  for sentence in tokens:
    sentence_length = len(sentence)
    for idx, center_word in enumerate(sentence):
      window_start = max(0, idx - window_size)
      window_end = min(sentence_length, idx + window_size + 1)
      center_word = sentence[idx]
      context_words = sentence[window_start:idx] + sentence[idx + 1:window_end]
      for context_word in context_words:
        pairs.append((center_word, context_word))
  return pairs

pairs = get_words_pairs(tokens, window_size=2)
print(pairs[:5])

[('굳', 'ㅋ'), ('ㅋ', '굳'), ('뭐', '야'), ('뭐', '이'), ('야', '뭐')]


In [ ]:
# 6.7 인덱스 쌍 변환
def get_index_pairs(word_pairs, token_to_id):# get_word_pairs 함수에서 생성된 단어 쌍을 토큰 인덱스 쌍으로 변환
  pairs = []
  unk_index = token_to_id["<unk>"]
  for word_pair in word_pairs :
    center_word, context_word = word_pair
    center_index = token_to_id.get(center_word, unk_index)
    context_index = token_to_id.get(context_word, unk_index)
    pairs.append((center_index, context_index))
  return pairs

index_pairs = get_index_pairs(pairs, token_to_id)
print(index_pairs[:5])

[(595, 100), (100, 595), (77, 176), (77, 2), (176, 77)]


In [ ]:
# 6.8 데이터로더 적용
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
#index_pairs는 get_index_pairs 함수에서 생성된 중심 단어와 주변 단어 토큰의 인덱스 쌍으로 이루어진 리스트
center_indexes = index_pairs[:, 0]
context_indexs = index_pairs[:, 1]

dataset = TensorDataset(center_indexes, context_indexs)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
# 6.9 Skip-gram 모델 준비 작업
from torch import optim

device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size = len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device) # 단어 사전 크기만큼 클래스가 있는 분류 문제
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

In [ ]:
# 6.10 모델 학습
for epoch in range(10):
  cost = 0.0
  for input_ids, target_ids in dataloader:
    input_ids = input_ids.to(device)
    targets_ids = target_ids.to(device)

    logits = word2vec(input_ids)
    loss = criterion(logits, targets_ids)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    cost += loss.item()

  cost /= len(dataloader)
  print(f"Epoch : {epoch}, Cost : {cost}")

In [ ]:
print(word2vec)

VanillaSkipgram(
  (embedding_dim): Embedding(5001, 128)
  (linear): Linear(in_features=128, out_features=5001, bias=True)
)


In [ ]:
# 6.11 임베딩 값 추출
token_to_embedding = dict()
embedding_matrix = word2vec.embedding_dim.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding

index = 30
token = vocab[30]
token_embedding = token_to_embedding[token]
print(token) # 인덱스 30 값의 단어
print(token_embedding) # 임베딩 값으로 단어 간의 유사도 확인

연기
[-5.19832373e-01 -6.37744367e-01 -5.07636964e-01  5.50549090e-01
 -1.06970978e+00  5.90682738e-02 -1.00693953e+00 -7.53396809e-01
  4.85068142e-01 -6.80525482e-01 -3.43408912e-01 -9.18117404e-01
 -1.58854261e-01  4.48266566e-01  6.18938088e-01 -1.84297395e+00
 -5.94056606e-01 -1.49275696e+00  1.16123569e+00 -1.20129919e+00
 -1.37378240e+00  7.50482678e-01  6.53322399e-01 -4.54647332e-01
  8.74155760e-01  3.05682242e-01 -1.46852207e+00  1.11466980e+00
  2.11248407e-03  9.58323479e-01  6.61307335e-01  9.94718373e-01
  8.21235955e-01  1.23524594e+00  1.07322574e-01 -9.73626375e-01
 -1.03757751e+00 -7.55655527e-01  3.82359564e-01  1.11448526e+00
  7.44541228e-01 -1.47491679e-01  7.33634651e-01  9.71342087e-01
 -4.67621446e-01  9.92715180e-01 -4.36405420e-01 -7.29028761e-01
  8.73310626e-01 -5.93718112e-01 -4.50348884e-01 -6.48179770e-01
  6.46720767e-01  2.95186788e-02 -2.99743712e-02  5.61961412e-01
  4.74930555e-01  1.04537308e+00  9.37365830e-01  4.15325582e-01
 -9.49518979e-01  1.04

In [ ]:
# 6.12 단어 임베딩 유사도 계산
from numpy.linalg import norm

def cosine_similarity(a, b): # a는 임베딩 토큰을, b는 임베당 행렬을 의미한다
  cosine = np.dot(b,a)/(norm(b, axis=1)*norm(a))
  return cosine

def top_n_index(cosine_matrix, n): # 유사도 행렬을 내림차순으로 정렬해 어떤 단어가 가장 가까운 단어인지 반환
  closest_indexes = cosine_matrix.argsort()[::-1]
  top_n = closest_indexes[1:n+1]
  return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5개 단어")
for index in top_n :
  print(f"{id_to_token[index]}-유사도 : {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5개 단어
유령-유사도 : 0.3125
아치-유사도 : 0.3026
보고싶은-유사도 : 0.2990
kbs-유사도 : 0.2979
일상-유사도 : 0.2819


## Gensim

In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 13.5 MB/s  0:00:02


In [ ]:
# 6.13 Word2Vec 모델 학습
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences = tokens,
    vector_size = 128,
    window = 5,
    min_count = 1,
    sg = 1,
   epochs =3,
    max_final_vocab = 10000
)

# word2vec.save("../models/word2vec.model")
# word2vec = Word2Vec.load("../models/word2vec.model")

In [ ]:
# 6.14 임베딩 추출 및 유사도 계산도
word = "연기"
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(w1=word, w2="연기"))

[-0.09053674 -0.30065066  0.19918491  0.49013495 -0.01307293 -0.0329362
  0.08501661  0.15242666 -0.7660679   0.33869106 -0.1587864  -0.37697926
 -0.08689267 -0.03857892 -0.29356158 -0.03346893 -0.34282055  0.44427314
 -0.32155     0.62469935  0.69015044  0.47617215 -0.2581496  -0.29991966
  0.05244838  0.00609967 -0.43041995  0.2716318   0.24974024  0.01979404
 -0.64245576  0.22646476  0.13357483 -0.0841282  -0.12580428 -0.42608505
  0.14269261 -0.13692962 -0.07557128 -0.46403524 -0.15419509  0.31638977
 -0.17436989 -0.38491216 -0.26959726  0.20644894 -0.27862543 -0.3986897
  0.26722518  0.0606314   0.7251796   0.2633002  -0.05267606  0.28718007
 -0.38035285 -0.33159816 -0.01925241  0.15510833 -0.4143577   0.09316771
  0.08018857 -0.06306428  0.01743544  0.12987906 -0.39321294  0.17858467
  0.1774011   0.30215818  0.18240687 -0.4143246  -0.23117968 -0.22657163
 -0.5715073   0.10843658 -0.1963276   0.11456671 -0.16804907 -0.35584405
 -0.06739822  0.32531384 -0.07336509 -0.13000078  0.1

## fastText

In [ ]:
# 6.15 KorNLI 데이터세트 전처리

corpus = Korpora.load("kornli")
corpus_texts = corpus.get_all_texts()+corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

In [ ]:
# 6.16 fastText 모델 실습

from gensim.models import FastText

fastText = FastText(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=5,
    sg=1,
    epochs=3,
    min_n=2,
    max_n=6)

In [ ]:
# 6.17 fastText OOV 처리
oov_token = "사랑해요"
oov_vector = fastText.wv(oov_token)
print(oov_vector in fastText.wv.indexx_to_key)
print(fastText.wv.most_similar(oov_token, topn=5))

TypeError: 'FastTextKeyedVectors' object is not callable

## RRN

In [ ]:
# 6.18 BiRNN

import torch
from torch import nn
input_size = 128
ouput_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(
    input_size=input_size,
    hidden_size=ouput_size,
    num_layers=num_layers,
    nonlinearity="tanh",
    batch_first=True,
    bidirectional=bidirectional,
)

batch_size = 4
sequence_len = 6
inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, ouput_size)
outputs, hidden = model(inputs, h_0)
print(outputs.shape)
print (hidden.shape)